<a href="https://colab.research.google.com/github/vaidiknakrani/parul_AI_ML_Learning/blob/main/Day6_Notebook2_IntelImages_CNN_TransferLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 6 — Notebook 2: Image Classification with CNNs
## Slides 10--11: Convolution · Pooling · ResNet · Transfer Learning

**Parul University & TelcoLearn | AI-ML Training Program 2027**  
**Day 6 | July 18, 2025 | 9:00 AM - 12:30 PM**

---

## What you will build

| Section | What you build | What you learn |
|---------|---------------|----------------|
| 1 | Convolution from scratch | What a filter actually does to pixels |
| 2 | Small CNN from scratch | Conv + Pool + Dense pipeline |
| 3 | Full CNN in Keras | BatchNorm, Dropout, callbacks |
| 4 | Transfer learning | Freeze MobileNetV2, train head only |
| 5 | Fine-tuning | Unfreeze top layers, low LR |
| 6 | Compare all three | Scratch vs Feature Extraction vs Fine-Tune |

**Dataset:** Intel Image Classification  
**Kaggle path:** `/kaggle/input/intel-image-classification`  
**Dataset URL:** https://www.kaggle.com/datasets/puneet6060/intel-image-classification  
**Task:** 6-class scene classification (buildings, forest, glacier, mountain, sea, street)  
**Size:** ~14,000 training images, 150x150 pixels  
**GPU:** Enable T4 GPU in Kaggle settings (Accelerator → GPU T4)

---
> **Key insight:** Transfer learning from ImageNet allows you to reach 90%+ accuracy
> on a 14k-image dataset in minutes. Training from scratch would take hours and reach
> lower accuracy without data augmentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, pathlib, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
if tf.config.list_physical_devices('GPU'):
    print(f'GPU: {tf.config.list_physical_devices("GPU")[0]}')
print('All imports successful.')

## Section 1 — Convolution from Scratch

Before using `Conv2D`, understand what it actually does.
A filter (kernel) slides across the image, computing a dot product at each position.

The formula from the slide:
$$(I * K)_{ij} = \sum_{m=0}^{f-1}\sum_{n=0}^{f-1} I_{(i+m)(j+n)} \cdot K_{mn}$$

In [ ]:
def convolve2d(image, kernel, padding='valid'):
    """2D convolution from scratch. Demonstrates what Conv2D does."""
    H, W   = image.shape
    f      = kernel.shape[0]  # assume square kernel
    if padding == 'same':
        p     = f // 2
        image = np.pad(image, p, mode='constant')
        H_out, W_out = H, W
    else:
        H_out = H - f + 1
        W_out = W - f + 1
    output = np.zeros((H_out, W_out))
    for i in range(H_out):
        for j in range(W_out):
            output[i, j] = (image[i:i+f, j:j+f] * kernel).sum()
    return output

# Create a simple test image (8x8 checkerboard)
img = np.zeros((8, 8))
img[::2, ::2] = 1; img[1::2, 1::2] = 1  # checkerboard

# Three classic filters
edge_h = np.array([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=float)  # Sobel horizontal
edge_v = np.array([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=float)  # Sobel vertical
blur   = np.ones((3,3), dtype=float) / 9                        # Average blur

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
results_conv = [
    ('Original', img, 'gray'),
    ('Edge (horizontal)', convolve2d(img, edge_h), 'RdBu'),
    ('Edge (vertical)',   convolve2d(img, edge_v), 'RdBu'),
    ('Blur',             convolve2d(img, blur),    'gray'),
]
for ax, (title, out, cmap) in zip(axes, results_conv):
    ax.imshow(out, cmap=cmap); ax.set_title(title, fontweight='bold')
    ax.axis('off')
plt.suptitle('What Convolution Filters Do', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Output size formula demonstration
n, f, p, s = 224, 3, 1, 1
out_size = (n - f + 2*p) // s + 1
print(f'Input: {n}x{n}, Kernel: {f}x{f}, Padding: {p}, Stride: {s}')
print(f'Output size: ({n} - {f} + 2*{p}) / {s} + 1 = {out_size}x{out_size}  (Same padding preserves size)')
out_valid = (n - f) // 1 + 1
print(f'Without padding (Valid): {out_valid}x{out_valid}  (shrinks by {n-out_valid} pixels each dim)')

In [ ]:
# ── Load Intel Image Classification Dataset ─────────────────────────────────
DATASET_PATH = '/kaggle/input/intel-image-classification'
# DATASET_PATH = '.'  # local run

IMG_SIZE  = (150, 150)
BATCH     = 32
CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

train_ds = keras.utils.image_dataset_from_directory(
    f'{DATASET_PATH}/seg_train/seg_train',
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='int', seed=42)

val_ds = keras.utils.image_dataset_from_directory(
    f'{DATASET_PATH}/seg_test/seg_test',
    image_size=IMG_SIZE, batch_size=BATCH, label_mode='int', seed=42)

# Prefetch for performance
AUTOTUNE  = tf.data.AUTOTUNE
train_ds  = train_ds.prefetch(AUTOTUNE)
val_ds    = val_ds.prefetch(AUTOTUNE)

# Visualise sample images
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for images, labels in train_ds.take(1):
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(images[i].numpy().astype('uint8'))
        ax.set_title(CLASS_NAMES[labels[i].numpy()], fontsize=9, fontweight='bold')
        ax.axis('off')
plt.suptitle('Intel Image Classification -- Sample Images', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Section 2 — Build CNN from Scratch

Three Conv+Pool blocks, GlobalAveragePooling, Dense head.
This mirrors the VGG architecture philosophy.

In [ ]:
def build_cnn_scratch(num_classes=6):
    """Small CNN from scratch. No pre-trained weights."""
    inputs = keras.Input(shape=(150, 150, 3))

    # Normalise pixels from [0,255] to [0,1]
    x = layers.Rescaling(1./255)(inputs)

    # Data augmentation (only applied during training)
    x = layers.RandomFlip('horizontal')(x)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)

    # Block 1: 32 filters, 3x3 kernel, Same padding
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)   # 150 -> 75

    # Block 2: 64 filters
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)   # 75 -> 37

    # Block 3: 128 filters
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)   # 37 -> 18

    # GlobalAveragePooling replaces Flatten+Dense: much fewer parameters
    # Collapses each 18x18 feature map to a single number
    x = layers.GlobalAveragePooling2D()(x)  # (batch, 128)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='CNN_from_scratch')
    return model

cnn_scratch = build_cnn_scratch()
cnn_scratch.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

print(cnn_scratch.summary())
print(f'Total parameters: {cnn_scratch.count_params():,}')

In [ ]:
callbacks_scratch = [
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=1),
]

print('Training CNN from scratch...')
hist_scratch = cnn_scratch.fit(
    train_ds, validation_data=val_ds,
    epochs=30, callbacks=callbacks_scratch, verbose=1)

# Plot
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['loss'],     'steelblue', label='Train', linewidth=2)
    axes[0].plot(history.history['val_loss'], 'darkorange', label='Val', linewidth=2)
    axes[0].set_title(f'{title} - Loss', fontweight='bold'); axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(history.history['accuracy'],     'steelblue', label='Train', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], 'darkorange', label='Val', linewidth=2)
    axes[1].set_title(f'{title} - Accuracy', fontweight='bold'); axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

plot_history(hist_scratch, 'CNN from Scratch')
acc_scratch = max(hist_scratch.history['val_accuracy'])
print(f'Best Val Accuracy (from scratch): {acc_scratch:.4f}')

## Section 3 — Transfer Learning: Feature Extraction

Load MobileNetV2 pre-trained on ImageNet. Freeze all its layers.
Only train the new classification head we add on top.

**Why MobileNetV2?** It's efficient, small, and already knows edges/shapes/objects
from 1.2 million ImageNet images. Our 14k images are not enough to learn this from scratch.

In [ ]:
# Load MobileNetV2 pre-trained on ImageNet, without the top classification layer
base_model = keras.applications.MobileNetV2(
    input_shape=(150, 150, 3),
    include_top=False,    # remove ImageNet classification head
    weights='imagenet'    # use ImageNet pre-trained weights
)

# FREEZE all base model layers -- we don't want to destroy pre-trained features
base_model.trainable = False
print(f'MobileNetV2 layers: {len(base_model.layers)}')
print(f'Trainable params (base frozen): {sum(v.numpy().size for v in base_model.trainable_weights):,}')

# Build the feature extraction model
inputs   = keras.Input(shape=(150, 150, 3))
x        = layers.Rescaling(1./255)(inputs)              # normalise
x        = layers.RandomFlip('horizontal')(x)             # augment
x        = layers.RandomRotation(0.1)(x)
x        = base_model(x, training=False)                  # frozen feature extractor
x        = layers.GlobalAveragePooling2D()(x)             # pool features
x        = layers.Dropout(0.3)(x)
x        = layers.Dense(128, activation='relu')(x)        # new trainable head
outputs  = layers.Dense(6, activation='softmax')(x)       # 6 scene classes

model_fe = keras.Model(inputs, outputs, name='MobileNetV2_FeatureExtraction')
model_fe.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])

print(f'Total params:     {model_fe.count_params():,}')
print(f'Trainable params: {sum(v.numpy().size for v in model_fe.trainable_weights):,}')
print('(Only the new head is trainable -- base is frozen)')

In [ ]:
callbacks_fe = [
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
]

print('Training: Feature Extraction (frozen MobileNetV2 base)...')
hist_fe = model_fe.fit(
    train_ds, validation_data=val_ds,
    epochs=15, callbacks=callbacks_fe, verbose=1)

plot_history(hist_fe, 'Transfer Learning: Feature Extraction')
acc_fe = max(hist_fe.history['val_accuracy'])
print(f'Best Val Accuracy (feature extraction): {acc_fe:.4f}')

## Section 4 — Fine-Tuning: Unfreeze Top Layers

After training the head, unfreeze the top 30 layers of MobileNetV2
and fine-tune everything together with a **very small learning rate**.

Large LR would destroy the pre-trained weights. Small LR gently adjusts them.

In [ ]:
# Unfreeze the entire base model
base_model.trainable = True

# But only fine-tune the top 30 layers -- keep early layers frozen
# Early layers learn basic edges/textures that transfer to any domain
# Later layers learn more domain-specific features
for layer in base_model.layers[:-30]:
    layer.trainable = False

unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen layers: {unfrozen} out of {len(base_model.layers)}')
print(f'Trainable params now: {sum(v.numpy().size for v in model_fe.trainable_weights):,}')

# Recompile with a much smaller learning rate
model_fe.compile(
    optimizer=keras.optimizers.Adam(1e-5),  # 100x smaller than before!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

callbacks_ft = [
    keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, verbose=1),
]

print('Fine-tuning: top 30 layers unfrozen, LR=1e-5...')
hist_ft = model_fe.fit(
    train_ds, validation_data=val_ds,
    epochs=20, callbacks=callbacks_ft, verbose=1)

plot_history(hist_ft, 'Fine-Tuning')
acc_ft = max(hist_ft.history['val_accuracy'])
print(f'Best Val Accuracy (fine-tuned): {acc_ft:.4f}')

In [ ]:
# ── Final comparison ────────────────────────────────────────────────────────
print('='*60)
print('  DAY 6 NOTEBOOK 2 - CNN + TRANSFER LEARNING RESULTS')
print('  Dataset: Intel Image Classification (6 classes, ~14k images)')
print('='*60)
print(f'  CNN from scratch       : Val Acc = {acc_scratch:.4f}')
print(f'  Feature extraction     : Val Acc = {acc_fe:.4f}')
print(f'  Fine-tuning            : Val Acc = {acc_ft:.4f}')
print()
# Visualise predictions on a batch
for images, labels in val_ds.take(1):
    preds = model_fe.predict(images, verbose=0)
    pred_classes = preds.argmax(axis=1)
    fig, axes = plt.subplots(3, 6, figsize=(18, 9))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(images[i].numpy().astype('uint8'))
        true_cls = CLASS_NAMES[labels[i].numpy()]
        pred_cls = CLASS_NAMES[pred_classes[i]]
        conf     = preds[i].max()
        color    = 'green' if true_cls == pred_cls else 'red'
        ax.set_title(f'True: {true_cls}\nPred: {pred_cls} ({conf:.2f})',
                     color=color, fontsize=8)
        ax.axis('off')
    plt.suptitle('Fine-tuned Model Predictions', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
print('Key lessons:')
print('  1. Convolution = sliding dot product. Detects local patterns.')
print('  2. Each Conv layer learns to detect: edges -> shapes -> objects.')
print('  3. GlobalAveragePooling eliminates millions of FC parameters.')
print('  4. Transfer learning reaches 90%+ with tiny data; scratch barely reaches 70%.')
print('  5. Fine-tuning with LR=1e-5 (not 1e-3!) gently adjusts pre-trained weights.')